# Facial Skincare 06: Strict Review-Only Core Query Generation

The production query is generated only from Notebook 05 query-safe target-review signals and the approved residual review text. Target item metadata is joined only after deterministic seed construction for leakage detection and removal. Insufficient selected cases are replaced from the same-regime eligible reserve when Notebook 05 evidence is available.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!pip install -q pyarrow dspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 25.5 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import html
import json
import re
import time

import dspy
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

In [5]:
# =========================================================
# Config
# =========================================================
PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")

SAMPLED_USERS_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_regime_sample.parquet"
ELIGIBLE_POOL_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_final_sampling_pool.parquet"
SAMPLED_USERS_MANIFEST_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_regime_sampling_manifest.json"
REVIEW_SIGNAL_PATH = PROJECT_ROOT / "data/processed/review_signals/face_review_signals.parquet"
ITEM_SCHEMA_PATH = PROJECT_ROOT / "data/processed/items/face_item_schema_full.parquet"

QUERY_CACHE_PATH = PROJECT_ROOT / "outputs/query_cache/face_queries.parquet"
QUERY_QC_PATH = PROJECT_ROOT / "outputs/query_summary/face_query_generation_qc.parquet"
QUERY_CONTRACT_PATH = PROJECT_ROOT / "outputs/query_summary/face_queries_config.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs/query_summary"

REGIME_ORDER = ["cold", "weak", "moderate", "strong"]
EXPECTED_ELIGIBLE_REGIME_COUNTS = None
EXPECTED_ELIGIBLE_ROWS = None
EXPECTED_INITIAL_TARGET_PER_REGIME = None
EXPECTED_INITIAL_OUTPUT_ROWS = None
EXPECTED_TARGET_PER_REGIME = None
EXPECTED_OUTPUT_ROWS = None
EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
MAX_TARGET_RANK_ALLOWED = 5
EXPECTED_EVALUATION_WINDOW_MONTHS = 9

QUERY_VARIANT = "medium_heavy_review_only_dspy_linguistic"
RANDOM_SEED = 42

MIN_QUERY_TOKENS = 5
MAX_QUERY_TOKENS = 18
TARGET_QUERY_TOKEN_MIN_FOR_SUMMARY = 8
TARGET_QUERY_TOKEN_MAX_FOR_SUMMARY = 18
MIN_SPECIFIC_FAMILIES = 2
MIN_SPECIFIC_SIGNALS = 3
TITLE_OVERLAP_WARNING_THRESHOLD = 0.45
MAX_RESIDUAL_TOKENS = 40
RESIDUAL_PAD_TARGET_TOKENS = TARGET_QUERY_TOKEN_MIN_FOR_SUMMARY
RESIDUAL_TEXT_COLUMN = "query_safe_residual_text"

RUN_SMOKE_TEST = False
SMOKE_TEST_N = 20
WRITE_OUTPUTS_IN_SMOKE_TEST = False

USE_DSPY_LINGUISTIC_REWRITE = True
FAIL_IF_DSPY_UNAVAILABLE = True
DEEPSEEK_MODEL = "openai/deepseek-chat"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
DEEPSEEK_COLAB_SECRET = "deepseek_api_key"
DSPY_TEMPERATURE = 0.0
DSPY_MAX_TOKENS = 160
DSPY_MAX_RETRIES = 2
DSPY_RETRY_SLEEP_SECONDS = 1.5
DSPY_MODE = "linguistic_rewrite_only"

FAMILY_SPECS = [
    ("product_type_or_form_texture", "qs_product_type_or_form_texture_signals", 2),
    ("concern", "qs_concern_signals", 2),
    ("benefit", "qs_benefit_signals", 2),
    ("skin_type", "qs_skin_type_signals", 1),
    ("ingredient", "qs_ingredient_signals", 2),
    ("usage_context", "qs_usage_context_signals", 1),
]

CATEGORY_ANCHOR_PHRASES = {
    "skin care", "skincare", "face care", "facial care", "face", "facial",
}
CATEGORY_ANCHOR_TOKENS = {"skin", "skincare", "face", "facial"}
GENERIC_UTILITY_TOKENS = {
    "support", "supports", "help", "helps", "promote", "promotes", "boost",
    "wellness", "natural", "formula", "blend", "complex", "routine", "care",
    "product", "products", "solution", "solutions",
}
CONTEXT_DEPENDENT_TOKENS = {"daily", "health"}
RATING_SENTIMENT_TOKENS = {
    "amazing", "awesome", "bad", "best", "better", "effective", "excellent",
    "favorite", "good", "great", "hate", "like", "love", "perfect",
    "recommend", "recommended", "terrible", "value", "wonderful", "works",
    "worked", "working", "rating", "rated", "star", "stars",
}
FUNCTION_WORDS = {
    "for", "with", "without", "and", "or", "to", "of", "in", "on", "as", "by",
    "is", "are", "that", "the", "a", "an",
}

METADATA_GENERIC_TOKENS = {
    "skin", "skincare", "care", "face", "facial", "serum", "cream", "gel",
    "lotion", "cleanser", "moisturizer", "mask", "masks", "treatment",
    "treatments", "toner", "essence", "beauty", "cosmetic", "cosmetics",
    "derm", "derma", "dermatology", "labs", "laboratory", "laboratories",
    "official", "store", "shop",
}
METADATA_NAME_COLUMNS = ["itemctx_facet_brand_text"]
METADATA_DIAGNOSTIC_COLUMNS = [
    "itemctx_identifier_diagnostic_text", "target_parent_asin",
]

TOKEN_EQUIVALENCE = {
    "sensitive": "sensitivity",
    "sensitivity": "sensitivity",
    "oily": "oiliness",
    "oiliness": "oiliness",
    "dry": "dryness",
    "dryness": "dryness",
    "wrinkle": "wrinkles",
    "wrinkles": "wrinkles",
    "hydrating": "hydration",
    "hydrate": "hydration",
    "hydration": "hydration",
    "hydrated": "hydration",
    "moisturizer": "moisturizer",
    "moisturizers": "moisturizer",
    "moisturizing": "moisturizer",
    "moisturize": "moisturizer",
    "cream": "moisturizer",
    "creams": "moisturizer",
    "lotion": "moisturizer",
    "lotions": "moisturizer",
    "serum": "serum",
    "serums": "serum",
    "mask": "mask",
    "masks": "mask",
    "cleanser": "cleanser",
    "cleansers": "cleanser",
    "cleansing": "cleanser",
    "wash": "cleanser",
    "toner": "toner",
    "toners": "toner",
    "treatment": "treatment",
    "treatments": "treatment",
    "acne": "acne",
    "blemish": "acne",
    "blemishes": "acne",
    "breakout": "acne",
    "breakouts": "acne",
    "pore": "pores",
    "pores": "pores",
    "brightening": "brightening",
    "brighten": "brightening",
    "brightness": "brightening",
    "hyperpigmentation": "dark_spots",
    "pigmentation": "dark_spots",
    "spots": "dark_spots",
    "spot": "dark_spots",
    "dark": "dark",
}

ASIN_PATTERN = re.compile(r"\bB0[A-Z0-9]{8}\b|\bB[0-9A-Z]{9}\b", re.IGNORECASE)
PACKAGE_PATTERN = re.compile(
    r"\b(\d+(?:\.\d+)?\s?(?:oz|fl\.?\s?oz|ml|g|gram|grams|ct|count|pack|packs|pcs|piece|pieces|%|percent|mg|mcg|iu)|spf\s?\d+|asin|seller|manufacturer|barcode|upc)\b",
    re.IGNORECASE,
)
SELLER_PATTERN = re.compile(
    r"\b(sold by|seller|manufacturer|made by|distributed by|shipped by)\b",
    re.IGNORECASE,
)

for path in [
    SAMPLED_USERS_PATH,
    ELIGIBLE_POOL_PATH,
    SAMPLED_USERS_MANIFEST_PATH,
    REVIEW_SIGNAL_PATH,
    ITEM_SCHEMA_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Input:", SAMPLED_USERS_PATH)
print("Input:", ELIGIBLE_POOL_PATH)
print("Input:", REVIEW_SIGNAL_PATH)
print("Input:", ITEM_SCHEMA_PATH)
print("Output:", QUERY_CACHE_PATH)


Input: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_user_regime_sample.parquet
Input: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_final_sampling_pool.parquet
Input: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/review_signals/face_review_signals.parquet
Input: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_schema_full.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet


In [6]:
# =========================================================
# Text and Evidence Helpers
# =========================================================
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text.replace("\n", " ").replace("\t", " ")).strip()


def to_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes"}


def to_int(value, default=0):
    numeric = pd.to_numeric(value, errors="coerce")
    return default if pd.isna(numeric) else int(numeric)


def tokenize(value):
    return re.findall(r"[a-z0-9']+", normalize_space(value).lower())


def token_count(value):
    return len(tokenize(value))


def split_pipe_values(value):
    text = normalize_space(value)
    if not text:
        return []
    values = []
    seen = set()
    for part in re.split(r"\s*\|\s*", text):
        cleaned = normalize_space(part).lower()
        if cleaned and cleaned not in seen:
            values.append(cleaned)
            seen.add(cleaned)
    return values


def pipe_join(values):
    out = []
    seen = set()
    for value in values:
        cleaned = normalize_space(value).lower()
        if cleaned and cleaned not in seen:
            out.append(cleaned)
            seen.add(cleaned)
    return " | ".join(out)


def normalize_content_token(token):
    return TOKEN_EQUIVALENCE.get(token, token)


def content_tokens(value):
    return {
        normalize_content_token(token)
        for token in tokenize(value)
        if token not in FUNCTION_WORDS
    }


def phrase_supported(query, phrase):
    phrase_concepts = content_tokens(phrase)
    return bool(phrase_concepts) and phrase_concepts.issubset(content_tokens(query))


def normalize_query_text(value):
    text = normalize_space(value).lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s\-']", " ", text)
    return re.sub(r"\s+", " ", text).strip(" -")


def split_metadata_values(value):
    text = normalize_space(value)
    if not text:
        return []
    return [
        normalize_space(part).lower()
        for part in re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", text)
        if normalize_space(part)
    ]


def remove_exact_phrase(text, phrase):
    phrase_tokens = tokenize(phrase)
    if not phrase_tokens:
        return normalize_query_text(text)
    pattern = r"(?<![a-z0-9])" + r"[\s_\-–—]+".join(
        re.escape(token) for token in phrase_tokens
    ) + r"(?:['’]s)?(?![a-z0-9])"
    return normalize_query_text(re.sub(pattern, " ", normalize_query_text(text), flags=re.IGNORECASE))


def exact_phrase_present(text, phrase):
    normalized = normalize_query_text(text)
    return bool(normalized) and remove_exact_phrase(normalized, phrase) != normalized


def is_specific_phrase(phrase):
    normalized = normalize_query_text(phrase)
    tokens = tokenize(normalized)
    if not tokens:
        return False
    if any(token in RATING_SENTIMENT_TOKENS for token in tokens):
        return False
    if normalized in CATEGORY_ANCHOR_PHRASES:
        return False
    if len(tokens) == 1 and tokens[0] in (
        CATEGORY_ANCHOR_TOKENS
        | GENERIC_UTILITY_TOKENS
        | CONTEXT_DEPENDENT_TOKENS
    ):
        return False
    specific_tokens = [
        token
        for token in tokens
        if token not in CATEGORY_ANCHOR_TOKENS
        and token not in GENERIC_UTILITY_TOKENS
        and token not in CONTEXT_DEPENDENT_TOKENS
        and token not in FUNCTION_WORDS
    ]
    return bool(specific_tokens)


def selected_family_phrases(row):
    family_map = {}
    for family, column, limit in FAMILY_SPECS:
        values = [
            value for value in split_pipe_values(row.get(column, ""))
            if is_specific_phrase(value)
        ]
        family_map[family] = values[:limit]
    return family_map


def selected_support_phrases(row):
    return [
        value
        for value in split_pipe_values(row.get("common_specific_support_phrases", ""))
        if is_specific_phrase(value)
    ]


def protected_specific_phrases(row):
    phrases = []
    for values in selected_family_phrases(row).values():
        phrases.extend(values)
    phrases.extend(selected_support_phrases(row))
    return list(dict.fromkeys(phrases))


def residual_content_tokens(value):
    blocked = (
        CATEGORY_ANCHOR_TOKENS
        | GENERIC_UTILITY_TOKENS
        | CONTEXT_DEPENDENT_TOKENS
        | RATING_SENTIMENT_TOKENS
        | FUNCTION_WORDS
    )
    out = []
    seen = set()
    for token in tokenize(normalize_query_text(value))[:MAX_RESIDUAL_TOKENS]:
        concept = normalize_content_token(token)
        if token.isdigit() or len(token) < 2 or token in blocked or concept in seen:
            continue
        out.append(token)
        seen.add(concept)
    return out


def build_review_safe_seed(row):
    family_map = selected_family_phrases(row)
    phrases = []
    for family, _, _ in FAMILY_SPECS:
        phrases.extend(family_map[family])
    phrases.extend(selected_support_phrases(row))
    phrases = list(dict.fromkeys(phrases))

    seed_tokens = []
    for phrase in phrases:
        phrase_tokens = tokenize(normalize_query_text(phrase))
        if not phrase_tokens:
            continue
        if len(seed_tokens) + len(phrase_tokens) > MAX_QUERY_TOKENS:
            continue
        seed_tokens.extend(phrase_tokens)

    seen_concepts = {normalize_content_token(token) for token in seed_tokens}
    for token in residual_content_tokens(row.get(RESIDUAL_TEXT_COLUMN, "")):
        if len(seed_tokens) >= RESIDUAL_PAD_TARGET_TOKENS:
            break
        concept = normalize_content_token(token)
        if concept in seen_concepts:
            continue
        seed_tokens.append(token)
        seen_concepts.add(concept)

    return normalize_query_text(" ".join(seed_tokens[:MAX_QUERY_TOKENS]))


def mask_protected_phrases(text, phrases):
    masked = normalize_query_text(text)
    for phrase in sorted(phrases, key=token_count, reverse=True):
        masked = remove_exact_phrase(masked, phrase)
    return masked


def generic_term_audit(text, row):
    masked = mask_protected_phrases(text, protected_specific_phrases(row))
    anchor_hits = []
    for phrase in sorted(CATEGORY_ANCHOR_PHRASES, key=len, reverse=True):
        if exact_phrase_present(masked, phrase):
            anchor_hits.append(phrase)
    anchor_hits.extend(
        token for token in tokenize(masked) if token in CATEGORY_ANCHOR_TOKENS
    )
    utility_hits = [
        token for token in tokenize(masked) if token in GENERIC_UTILITY_TOKENS
    ]
    return pipe_join(anchor_hits), pipe_join(utility_hits)


def query_specific_family_count(text, row):
    return sum(
        any(phrase_supported(text, phrase) for phrase in values)
        for values in selected_family_phrases(row).values()
    )


def missing_seed_multiword_phrases(seed_query, candidate_query, row):
    required = [
        phrase
        for phrase in protected_specific_phrases(row)
        if token_count(phrase) > 1 and exact_phrase_present(seed_query, phrase)
    ]
    return [
        phrase for phrase in required
        if not exact_phrase_present(candidate_query, phrase)
    ]


def prohibited_rating_sentiment_terms(text):
    return pipe_join(
        token for token in tokenize(text) if token in RATING_SENTIMENT_TOKENS
    )


def title_overlap_ratio(query, title):
    query_tokens = set(tokenize(query))
    title_tokens = {token for token in tokenize(title) if len(token) >= 4}
    if not query_tokens or not title_tokens:
        return 0.0
    return len(query_tokens & title_tokens) / len(query_tokens)


def distinctive_metadata_terms(value):
    terms = []
    for phrase in split_metadata_values(value):
        phrase_tokens = tokenize(phrase)
        if len(phrase_tokens) >= 2:
            terms.append(" ".join(phrase_tokens))
        terms.extend(
            token
            for token in phrase_tokens
            if len(token) >= 3 and token not in METADATA_GENERIC_TOKENS
        )
    return terms


def diagnostic_metadata_terms(value):
    terms = []
    for phrase in split_metadata_values(value):
        phrase_tokens = tokenize(phrase)
        if phrase_tokens:
            terms.append(" ".join(phrase_tokens))
        terms.extend(token for token in phrase_tokens if len(token) >= 3)
    return terms


def metadata_term_groups(row):
    name_terms = []
    for column in METADATA_NAME_COLUMNS:
        name_terms.extend(distinctive_metadata_terms(row.get(column, "")))

    identifier_terms = []
    for column in METADATA_DIAGNOSTIC_COLUMNS:
        identifier_terms.extend(diagnostic_metadata_terms(row.get(column, "")))

    title = normalize_query_text(row.get("itemctx_title", ""))
    return {
        "name_terms": list(dict.fromkeys(name_terms)),
        "identifier_terms": list(dict.fromkeys(identifier_terms)),
        "title": title,
    }


def scrub_target_metadata_cues(text, row):
    scrubbed = normalize_query_text(text)
    removed = []

    for label, pattern in [
        ("asin_pattern", ASIN_PATTERN),
        ("package_or_dosage_pattern", PACKAGE_PATTERN),
        ("seller_or_manufacturer_pattern", SELLER_PATTERN),
    ]:
        updated = normalize_query_text(pattern.sub(" ", scrubbed))
        if updated != scrubbed:
            removed.append(label)
        scrubbed = updated

    groups = metadata_term_groups(row)
    for term in groups["name_terms"] + groups["identifier_terms"]:
        updated = remove_exact_phrase(scrubbed, term)
        if updated != scrubbed:
            removed.append(term)
        scrubbed = updated

    title = groups["title"]
    if token_count(title) >= 2:
        updated = remove_exact_phrase(scrubbed, title)
        if updated != scrubbed:
            removed.append("exact_title_phrase")
        scrubbed = updated

    return normalize_query_text(scrubbed), pipe_join(removed)


def metadata_leakage_flags(text, row):
    normalized = normalize_query_text(text)
    groups = metadata_term_groups(row)
    title = groups["title"]
    return {
        "brand_or_name_leak_flag": int(any(
            exact_phrase_present(normalized, term) for term in groups["name_terms"]
        )),
        "identifier_like_leak_flag": int(any(
            exact_phrase_present(normalized, term) for term in groups["identifier_terms"]
        )),
        "asin_leak_flag": int(bool(ASIN_PATTERN.search(normalized))),
        "package_cue_flag": int(bool(PACKAGE_PATTERN.search(normalized))),
        "seller_manufacturer_leak_flag": int(bool(SELLER_PATTERN.search(normalized))),
        "exact_title_phrase_flag": int(
            token_count(title) >= 2 and exact_phrase_present(normalized, title)
        ),
        "title_overlap_ratio": float(title_overlap_ratio(normalized, title)),
    }


def blocking_leakage_count(flags):
    return sum(
        flags[column]
        for column in [
            "brand_or_name_leak_flag",
            "identifier_like_leak_flag",
            "asin_leak_flag",
            "package_cue_flag",
            "seller_manufacturer_leak_flag",
            "exact_title_phrase_flag",
        ]
    )


def evidence_reason(row):
    reasons = []
    if "signal_available" in row.index and not to_bool(row.get("signal_available")):
        reasons.append("missing_notebook05_signal")
    if to_bool(row.get("signal_available", True)) and not to_bool(
        row.get("review_only_query_evidence_sufficient")
    ):
        reasons.append("upstream_review_only_evidence_insufficient")
    if to_int(row.get("seed_token_count")) < MIN_QUERY_TOKENS:
        reasons.append("review_safe_seed_too_short")
    if to_int(row.get("seed_specific_family_count")) < MIN_SPECIFIC_FAMILIES:
        reasons.append("specific_family_count_below_threshold")
    if to_int(row.get("seed_source_signal_total_count")) < MIN_SPECIFIC_SIGNALS:
        reasons.append("specific_signal_total_below_threshold")
    if normalize_space(row.get("seed_generic_anchor_terms")):
        reasons.append("generic_category_anchor_remaining")
    if normalize_space(row.get("seed_generic_utility_terms")):
        reasons.append("generic_utility_remaining")
    if normalize_space(row.get("seed_rating_sentiment_terms")):
        reasons.append("rating_or_sentiment_term_remaining")
    if to_int(row.get("seed_blocking_leakage_flag_count")) > 0:
        reasons.append("target_metadata_leakage_remaining")
    return " | ".join(reasons)


def bool_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return series.fillna("").astype(str).str.strip().str.lower().isin({"true", "1", "yes"})

In [7]:
# =========================================================
# Load and Validate Notebook 03 and 05 Artifacts
# =========================================================
sampled_columns = [
    "case_id", "user_id", "parent_asin", "target_timestamp_ms", "regime",
    "sampling_bracket", "target_selection_mode", "sampled_regime_rank",
    "selection_rank_within_regime", "target_rank_desc",
]
eligible_columns = [
    "case_id", "user_id", "parent_asin", "review_timestamp_ms", "regime",
    "sampling_bracket", "target_selection_mode", "target_rank_desc",
]

sampled_users_df = pd.read_parquet(SAMPLED_USERS_PATH, columns=sampled_columns)
eligible_pool_df = pd.read_parquet(ELIGIBLE_POOL_PATH, columns=eligible_columns)
review_signal_df = pd.read_parquet(REVIEW_SIGNAL_PATH)

with open(SAMPLED_USERS_MANIFEST_PATH, "r", encoding="utf-8") as file:
    sampled_manifest = json.load(file)

required_signal_columns = [
    "case_id", "user_id", "target_parent_asin", "regime",
    "query_evidence_source", "item_metadata_evidence_used",
    "historical_review_evidence_used", "user_prior_evidence_used",
    "rating_evidence_used", "sentiment_evidence_used",
    "query_safe_residual_text", "common_specific_support_phrases",
    "common_specific_signal_seed_text", "query_safe_signal_family_count",
    "query_safe_signal_total_count", "review_only_query_evidence_sufficient",
] + [column for _, column, _ in FAMILY_SPECS]
missing_signal_columns = sorted(set(required_signal_columns) - set(review_signal_df.columns))
if missing_signal_columns:
    raise RuntimeError(f"Notebook 05 output is missing columns: {missing_signal_columns}")

for frame_name, frame in {
    "sampled users": sampled_users_df,
    "eligible pool": eligible_pool_df,
    "review signals": review_signal_df,
}.items():
    frame["case_id"] = frame["case_id"].astype(str).str.strip()
    if frame["case_id"].eq("").any() or frame["case_id"].duplicated().any():
        raise RuntimeError(f"{frame_name} must contain unique non-empty case_id values.")

sampled_users_df["parent_asin"] = sampled_users_df["parent_asin"].astype(str).str.strip()
eligible_pool_df["parent_asin"] = eligible_pool_df["parent_asin"].astype(str).str.strip()
eligible_pool_df["target_timestamp_ms"] = pd.to_numeric(
    eligible_pool_df["review_timestamp_ms"], errors="coerce"
)
if eligible_pool_df["target_timestamp_ms"].isna().any():
    raise RuntimeError("Eligible pool contains missing target timestamps.")

if sampled_users_df["user_id"].duplicated().any():
    raise RuntimeError("Initially sampled users must be unique.")
if eligible_pool_df["user_id"].duplicated().any():
    raise RuntimeError("Eligible reserve pool must contain one case per user.")

observed_eligible_counts = (
    eligible_pool_df["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict()
)
EXPECTED_ELIGIBLE_REGIME_COUNTS = dict(observed_eligible_counts)
EXPECTED_ELIGIBLE_ROWS = int(len(eligible_pool_df))

observed_sample_counts = (
    sampled_users_df["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int)
)
if len(set(observed_sample_counts.to_dict().values())) != 1:
    raise RuntimeError(f"Initial regime quota must be balanced: {observed_sample_counts.to_dict()}")
EXPECTED_INITIAL_TARGET_PER_REGIME = int(observed_sample_counts.iloc[0])
EXPECTED_INITIAL_OUTPUT_ROWS = int(len(sampled_users_df))
if not sampled_users_df["target_selection_mode"].eq(EXPECTED_TARGET_SELECTION_MODE).all():
    raise RuntimeError("Initial target_selection_mode mismatch.")
if not eligible_pool_df["target_selection_mode"].eq(EXPECTED_TARGET_SELECTION_MODE).all():
    raise RuntimeError("Eligible-pool target_selection_mode mismatch.")
if not sampled_users_df["target_rank_desc"].between(1, MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"Initial target_rank_desc must be between 1 and {MAX_TARGET_RANK_ALLOWED}.")
if not eligible_pool_df["target_rank_desc"].between(1, MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"Eligible-pool target_rank_desc must be between 1 and {MAX_TARGET_RANK_ALLOWED}.")
if sampled_manifest.get("target_selection_mode") != EXPECTED_TARGET_SELECTION_MODE:
    raise RuntimeError("Notebook 03 manifest target_selection_mode mismatch.")
if int(sampled_manifest.get("evaluation_window_months", -1)) != EXPECTED_EVALUATION_WINDOW_MONTHS:
    raise RuntimeError("Notebook 03 manifest evaluation_window_months mismatch.")
if int(sampled_manifest.get("max_target_rank_allowed", -1)) != MAX_TARGET_RANK_ALLOWED:
    raise RuntimeError("Notebook 03 manifest max_target_rank_allowed mismatch.")

if not review_signal_df["query_evidence_source"].eq("target_review_only").all():
    raise RuntimeError("Notebook 05 query_evidence_source must be target_review_only.")
for column in [
    "item_metadata_evidence_used", "historical_review_evidence_used",
    "user_prior_evidence_used", "rating_evidence_used", "sentiment_evidence_used",
]:
    if bool_series(review_signal_df[column]).any():
        raise RuntimeError(f"Notebook 05 evidence flag must be false: {column}")

forbidden_signal_columns = {
    "target_review_text", "heldout_review_text", "review_text", "review_body",
    "raw_review_text", "prior_review_text", "prior_history_n", "prior_review_n",
    "rating", "sentiment", "prompt", "response", "llm_response",
    "query_safe_facet_text", "common_functional_facet_text",
    "historical_review_reputation_text", "review_reputation_facet_text",
}
forbidden_signal_columns_present = sorted(forbidden_signal_columns & set(review_signal_df.columns))
if forbidden_signal_columns_present:
    raise RuntimeError(
        f"Forbidden query-evidence columns are present in Notebook 05 output: {forbidden_signal_columns_present}"
    )

eligible_case_ids = set(eligible_pool_df["case_id"])
selected_case_ids = set(sampled_users_df["case_id"])
signal_case_ids = set(review_signal_df["case_id"])
if not selected_case_ids.issubset(eligible_case_ids):
    raise RuntimeError("Initial sample contains cases outside the eligible pool.")
if len(review_signal_df) != EXPECTED_ELIGIBLE_ROWS:
    raise RuntimeError(
        f"Notebook 05 must cover the full Facial eligible pool: {len(review_signal_df)}/{EXPECTED_ELIGIBLE_ROWS}."
    )
if signal_case_ids != eligible_case_ids:
    missing_signal_case_ids = eligible_case_ids - signal_case_ids
    extra_signal_case_ids = signal_case_ids - eligible_case_ids
    raise RuntimeError(
        "Notebook 05 case_id set must equal the Notebook 03 eligible pool: "
        f"missing={len(missing_signal_case_ids)}, extra={len(extra_signal_case_ids)}."
    )

print("Rows: eligible", len(eligible_pool_df))
print("Rows: initial", len(sampled_users_df))
print("Rows: signals", len(review_signal_df))
print("Validation: upstream contracts passed")


Rows: eligible 28348
Rows: initial 2792
Rows: signals 28348
Validation: upstream contracts passed


In [8]:
# =========================================================
# Build Review-Only Seeds and Metadata Leakage Audit
# =========================================================
seed_records = []
for row in review_signal_df.itertuples(index=False):
    row_dict = row._asdict()
    seed_records.append({
        "case_id": str(row_dict["case_id"]),
        "review_safe_seed_before_metadata_audit": build_review_safe_seed(row_dict),
    })
seed_df = pd.DataFrame(seed_records)
review_signal_seed_df = review_signal_df.merge(seed_df, on="case_id", how="left", validate="one_to_one")
review_signal_seed_df["target_parent_asin"] = review_signal_seed_df["target_parent_asin"].astype(str).str.strip()

item_context_required_cols = ["parent_asin", "title", "facet_brand_text"]
item_context_optional_cols = ["identifier_diagnostic_text"]

item_schema_available_cols = set(pd.read_parquet(ITEM_SCHEMA_PATH, columns=[]).columns)

item_context_read_cols = item_context_required_cols + [
    col for col in item_context_optional_cols
    if col in item_schema_available_cols
]

item_context_df = pd.read_parquet(
    ITEM_SCHEMA_PATH,
    columns=item_context_read_cols,
)

for col in item_context_optional_cols:
    if col not in item_context_df.columns:
        item_context_df[col] = ""

item_context_df["parent_asin"] = item_context_df["parent_asin"].astype(str).str.strip()
if item_context_df["parent_asin"].eq("").any() or item_context_df["parent_asin"].duplicated().any():
    raise RuntimeError("Item schema audit keys must be unique and non-empty.")
item_context_df = item_context_df.rename(columns={
    "parent_asin": "target_parent_asin",
    "title": "itemctx_title",
    "facet_brand_text": "itemctx_facet_brand_text",
    "identifier_diagnostic_text": "itemctx_identifier_diagnostic_text",
})

audit_df = review_signal_seed_df.merge(
    item_context_df,
    on="target_parent_asin",
    how="left",
    validate="many_to_one",
    indicator=True,
)
if not audit_df["_merge"].eq("both").all():
    raise RuntimeError("Target metadata audit join did not match every Notebook 05 signal row.")
audit_df = audit_df.drop(columns="_merge")
for column in ["itemctx_title", "itemctx_facet_brand_text", "itemctx_identifier_diagnostic_text"]:
    audit_df[column] = audit_df[column].fillna("").astype(str)

candidate_records = []
for row in audit_df.itertuples(index=False):
    row_dict = row._asdict()
    scrubbed_seed, removed_terms = scrub_target_metadata_cues(
        row_dict["review_safe_seed_before_metadata_audit"], row_dict
    )
    leak_flags = metadata_leakage_flags(scrubbed_seed, row_dict)
    generic_anchors, generic_utilities = generic_term_audit(scrubbed_seed, row_dict)
    family_count = query_specific_family_count(scrubbed_seed, row_dict)
    rating_sentiment_terms = prohibited_rating_sentiment_terms(scrubbed_seed)
    blocking_count = blocking_leakage_count(leak_flags)
    source_signal_total_count = to_int(row_dict.get("query_safe_signal_total_count"))
    sufficient = bool(
        to_bool(row_dict.get("review_only_query_evidence_sufficient"))
        and token_count(scrubbed_seed) >= MIN_QUERY_TOKENS
        and family_count >= MIN_SPECIFIC_FAMILIES
        and source_signal_total_count >= MIN_SPECIFIC_SIGNALS
        and not generic_anchors
        and not generic_utilities
        and not rating_sentiment_terms
        and blocking_count == 0
    )
    candidate_records.append({
        "case_id": str(row_dict["case_id"]),
        "review_safe_seed": scrubbed_seed,
        "seed_token_count": token_count(scrubbed_seed),
        "seed_specific_family_count": int(family_count),
        "seed_source_signal_total_count": int(source_signal_total_count),
        "seed_generic_anchor_terms": generic_anchors,
        "seed_generic_utility_terms": generic_utilities,
        "seed_rating_sentiment_terms": rating_sentiment_terms,
        "seed_metadata_removed_terms": removed_terms,
        "seed_blocking_leakage_flag_count": int(blocking_count),
        "seed_title_overlap_ratio": float(leak_flags["title_overlap_ratio"]),
        "query_candidate_sufficient": sufficient,
    })

candidate_eval_df = audit_df.merge(
    pd.DataFrame(candidate_records), on="case_id", how="left", validate="one_to_one"
)
candidate_eval_df["signal_available"] = True
candidate_eval_df["insufficient_reason"] = candidate_eval_df.apply(evidence_reason, axis=1)

pool_df = eligible_pool_df.merge(
    candidate_eval_df,
    on="case_id",
    how="left",
    suffixes=("", "_signal"),
    validate="one_to_one",
)
pool_df["signal_available"] = pool_df["signal_available"].fillna(False).astype(bool)
pool_df["query_candidate_sufficient"] = pool_df["query_candidate_sufficient"].fillna(False).astype(bool)
pool_df["initial_selected"] = pool_df["case_id"].isin(selected_case_ids)
pool_df["insufficient_reason"] = pool_df.apply(evidence_reason, axis=1)

matched_targets = pool_df.loc[pool_df["signal_available"], ["parent_asin", "target_parent_asin"]]
if not matched_targets["parent_asin"].astype(str).eq(matched_targets["target_parent_asin"].astype(str)).all():
    raise RuntimeError("Notebook 03 and Notebook 05 target item ids do not match.")

print("Rows: sufficient signal cases", int(pool_df["query_candidate_sufficient"].sum()))
print("Validation: review-only seed audit passed")

Rows: sufficient signal cases 20742
Validation: review-only seed audit passed


In [9]:
# =========================================================
# Same-Regime Deterministic Replacement
# =========================================================
strict_feasible_counts = (
    pool_df.loc[pool_df["query_candidate_sufficient"], "regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)
strict_feasible_target_per_regime = int(strict_feasible_counts.min())
EXPECTED_TARGET_PER_REGIME = int(strict_feasible_target_per_regime)
EXPECTED_OUTPUT_ROWS = int(EXPECTED_TARGET_PER_REGIME * len(REGIME_ORDER))
if EXPECTED_TARGET_PER_REGIME <= 0:
    raise RuntimeError(
        "No strict review-only balanced query quota is feasible: "
        f"counts={strict_feasible_counts.to_dict()}"
    )

print("Strict feasible counts:", strict_feasible_counts.to_dict())
print("Dynamic strict target per regime:", EXPECTED_TARGET_PER_REGIME)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

insufficient_audit_df = pool_df.loc[
    (pool_df["signal_available"] & ~pool_df["query_candidate_sufficient"])
    | (pool_df["initial_selected"] & ~pool_df["signal_available"]),
    [
        "case_id", "user_id", "parent_asin", "regime", "initial_selected",
        "signal_available", "review_only_query_evidence_sufficient",
        "query_safe_signal_family_count", "query_safe_signal_total_count",
        "seed_token_count", "seed_specific_family_count", "seed_generic_anchor_terms",
        "seed_generic_utility_terms", "seed_rating_sentiment_terms",
        "seed_blocking_leakage_flag_count", "insufficient_reason",
    ],
].copy()
insufficient_audit_df.to_csv(
    OUTPUT_DIR / "face_query_insufficient_review_evidence.csv",
    index=False,
    encoding="utf-8-sig",
)

sample_order_df = sampled_users_df.sort_values(
    ["regime", "selection_rank_within_regime", "case_id"]
).copy()
selection_plan_rows = []
replacement_rows = []
shortfalls = {}

for regime in REGIME_ORDER:
    initial_regime = sample_order_df[sample_order_df["regime"].eq(regime)].copy()
    initial_regime = initial_regime.sort_values(["selection_rank_within_regime", "case_id"])
    pool_regime = pool_df[pool_df["regime"].eq(regime)].copy()
    sufficient_by_case = pool_regime.set_index("case_id")["query_candidate_sufficient"].to_dict()

    initial_regime["query_candidate_sufficient"] = initial_regime["case_id"].map(sufficient_by_case).fillna(False).astype(bool)
    initial_sufficient = initial_regime[initial_regime["query_candidate_sufficient"]].copy()
    initial_insufficient_case_ids = (
        initial_regime.loc[~initial_regime["query_candidate_sufficient"], "case_id"]
        .astype(str)
        .tolist()
    )

    reserve_regime = pool_regime[
        ~pool_regime["case_id"].isin(initial_regime["case_id"])
        & pool_regime["query_candidate_sufficient"]
    ].sort_values("case_id")
    reserve_case_ids = reserve_regime["case_id"].astype(str).tolist()

    final_slot = 0
    selected_initial = initial_sufficient.head(EXPECTED_TARGET_PER_REGIME)
    for initial_row in selected_initial.itertuples(index=False):
        final_slot += 1
        initial_case_id = str(initial_row.case_id)
        selection_plan_rows.append({
            "regime": regime,
            "selection_slot": int(final_slot),
            "initial_case_id": initial_case_id,
            "final_case_id": initial_case_id,
            "replacement_case_used": False,
            "replacement_source_case_id": "",
            "replaced_case_id": "",
        })

    needed_replacements = EXPECTED_TARGET_PER_REGIME - final_slot
    available_replacements = min(needed_replacements, len(reserve_case_ids))

    for replacement_index in range(available_replacements):
        final_slot += 1
        final_case_id = reserve_case_ids[replacement_index]
        replaced_case_id = (
            initial_insufficient_case_ids[replacement_index]
            if replacement_index < len(initial_insufficient_case_ids)
            else ""
        )
        selection_plan_rows.append({
            "regime": regime,
            "selection_slot": int(final_slot),
            "initial_case_id": replaced_case_id,
            "final_case_id": final_case_id,
            "replacement_case_used": True,
            "replacement_source_case_id": final_case_id,
            "replaced_case_id": replaced_case_id,
        })
        replacement_rows.append({
            "regime": regime,
            "replaced_case_id": replaced_case_id,
            "replacement_source_case_id": final_case_id,
            "replacement_status": "replaced_from_same_regime_reserve",
        })

    if final_slot < EXPECTED_TARGET_PER_REGIME:
        shortfalls[regime] = EXPECTED_TARGET_PER_REGIME - final_slot
        replacement_rows.append({
            "regime": regime,
            "replaced_case_id": "",
            "replacement_source_case_id": "",
            "replacement_status": "reserve_shortfall",
        })

replacement_mapping_df = pd.DataFrame(replacement_rows, columns=[
    "regime", "replaced_case_id", "replacement_source_case_id", "replacement_status"
])
replacement_mapping_df.to_csv(
    OUTPUT_DIR / "face_query_replacement_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)

if shortfalls:
    signal_coverage = {
        regime: int(
            pool_df.loc[
                pool_df["regime"].eq(regime) & ~pool_df["initial_selected"],
                "signal_available",
            ].sum()
        )
        for regime in shortfalls
    }
    raise RuntimeError(
        "Same-regime reserve cannot satisfy the strict review-only quota. "
        f"Shortfalls: {shortfalls}. Reserve cases with Notebook 05 signals: {signal_coverage}. "
        "Run Notebook 05 for the required reserve cases; target metadata fallback is prohibited."
    )

selection_plan_df = pd.DataFrame(selection_plan_rows)
if selection_plan_df["final_case_id"].eq("").any():
    raise RuntimeError("Final selection plan contains an unfilled query slot.")
if selection_plan_df["final_case_id"].duplicated().any():
    raise RuntimeError("Replacement selection produced duplicate final case ids.")

final_selection_df = selection_plan_df.merge(
    pool_df,
    left_on="final_case_id",
    right_on="case_id",
    how="left",
    validate="one_to_one",
)
if not final_selection_df["query_candidate_sufficient"].all():
    raise RuntimeError("Final selection contains insufficient review-only evidence.")
if not final_selection_df["regime_x"].eq(final_selection_df["regime_y"]).all():
    raise RuntimeError("Replacement crossed regime boundaries.")
if final_selection_df["user_id"].duplicated().any():
    raise RuntimeError("Final replacement selection must retain unique users.")

final_counts = (
    final_selection_df["regime_x"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int)
)
if not final_counts.eq(EXPECTED_TARGET_PER_REGIME).all():
    raise RuntimeError(f"Final replacement quota mismatch: {final_counts.to_dict()}")

print("Rows: insufficient", len(insufficient_audit_df))
print("Rows: replacements", len(replacement_mapping_df))
print("Rows: final", len(final_selection_df))
print("Validation: replacement contract passed")


Strict feasible counts: {'cold': 16068, 'weak': 3166, 'moderate': 572, 'strong': 936}
Dynamic strict target per regime: 572
Rows: insufficient 7606
Rows: replacements 85
Rows: final 2288
Validation: replacement contract passed


In [10]:
# =========================================================
# DSPy Seed-Only Rewrite
# =========================================================
def get_deepseek_api_key():
    from google.colab import userdata
    return userdata.get(DEEPSEEK_COLAB_SECRET)


DSPY_PROGRAM = None
if USE_DSPY_LINGUISTIC_REWRITE:
    api_key = get_deepseek_api_key()
    if not api_key and FAIL_IF_DSPY_UNAVAILABLE:
        raise RuntimeError(
            f"DeepSeek API key is required in Colab secret {DEEPSEEK_COLAB_SECRET!r}."
        )
    if api_key:
        dspy.configure(
            lm=dspy.LM(
                model=DEEPSEEK_MODEL,
                api_key=api_key,
                api_base=DEEPSEEK_BASE_URL,
                temperature=DSPY_TEMPERATURE,
                max_tokens=DSPY_MAX_TOKENS,
            )
        )
        DSPY_PROGRAM = dspy.Predict("seed_query -> query")


def call_dspy(seed_query):
    if not USE_DSPY_LINGUISTIC_REWRITE:
        return seed_query, "dspy_disabled"
    if DSPY_PROGRAM is None:
        return seed_query, "dspy_unavailable"
    for attempt in range(1, DSPY_MAX_RETRIES + 1):
        prediction = DSPY_PROGRAM(seed_query=seed_query)
        rewritten = normalize_query_text(getattr(prediction, "query", ""))
        if rewritten:
            return rewritten, "dspy_linguistic_rewrite"
        if attempt < DSPY_MAX_RETRIES:
            time.sleep(DSPY_RETRY_SLEEP_SECONDS)
    return seed_query, "dspy_empty_output"


def rewrite_new_content_tokens(seed_query, candidate_query):
    return sorted(content_tokens(candidate_query) - content_tokens(seed_query))


def validate_generated_query(query, seed_query, row):
    scrubbed, removed_terms = scrub_target_metadata_cues(query, row)
    leak_flags = metadata_leakage_flags(scrubbed, row)
    generic_anchors, generic_utilities = generic_term_audit(scrubbed, row)
    family_count = query_specific_family_count(scrubbed, row)
    new_tokens = rewrite_new_content_tokens(seed_query, scrubbed)
    missing_multiword = missing_seed_multiword_phrases(seed_query, scrubbed, row)
    rating_sentiment_terms = prohibited_rating_sentiment_terms(scrubbed)
    valid = bool(
        MIN_QUERY_TOKENS <= token_count(scrubbed) <= MAX_QUERY_TOKENS
        and family_count >= MIN_SPECIFIC_FAMILIES
        and not generic_anchors
        and not generic_utilities
        and not rating_sentiment_terms
        and not new_tokens
        and not missing_multiword
        and blocking_leakage_count(leak_flags) == 0
    )
    return {
        "query": scrubbed,
        "valid": valid,
        "removed_terms": removed_terms,
        "generic_anchor_terms": generic_anchors,
        "generic_utility_terms": generic_utilities,
        "rating_sentiment_terms": rating_sentiment_terms,
        "specific_family_count": int(family_count),
        "new_content_tokens": new_tokens,
        "missing_multiword_phrases": missing_multiword,
        **leak_flags,
    }


def face_smoke_sample(frame, total_n):
    if total_n <= 0:
        raise RuntimeError("SMOKE_TEST_N must be positive.")
    return frame.sort_values(["regime_x", "selection_slot"]).head(total_n).copy()


generation_df = (
    face_smoke_sample(final_selection_df, SMOKE_TEST_N)
    if RUN_SMOKE_TEST
    else final_selection_df.sort_values(["regime_x", "selection_slot"]).reset_index(drop=True)
)

query_rows = []
qc_rows = []
for row in generation_df.itertuples(index=False):
    row_dict = row._asdict()
    seed_query = normalize_query_text(row_dict["review_safe_seed"])
    candidate_query, dspy_status = call_dspy(seed_query)
    candidate_audit = validate_generated_query(candidate_query, seed_query, row_dict)

    if candidate_audit["valid"]:
        final_audit = candidate_audit
        dspy_accepted = dspy_status == "dspy_linguistic_rewrite"
        query_source = "dspy_linguistic_rewrite" if dspy_accepted else "deterministic_review_safe_seed"
        fallback_used = not dspy_accepted
    else:
        final_audit = validate_generated_query(seed_query, seed_query, row_dict)
        if not final_audit["valid"]:
            raise RuntimeError(
                f"Prevalidated deterministic seed failed final validation for case {row_dict['case_id']}."
            )
        dspy_accepted = False
        query_source = "deterministic_review_safe_seed_fallback"
        fallback_used = True

    final_query = final_audit["query"]
    anchor_term_count = len(split_pipe_values(final_audit["generic_anchor_terms"]))

    query_row = {
        "case_id": str(row_dict["case_id"]),
        "user_id": str(row_dict["user_id"]),
        "parent_asin": str(row_dict["parent_asin"]),
        "target_parent_asin": str(row_dict["parent_asin"]),
        "target_timestamp_ms": int(row_dict["target_timestamp_ms"]),
        "regime": str(row_dict["regime_x"]),
        "profile_regime": str(row_dict["regime_x"]),
        "sampling_bracket": str(row_dict.get("sampling_bracket", "")),
        "target_selection_mode": str(row_dict["target_selection_mode"]),
        "safe_signal_count": to_int(row_dict.get("query_safe_signal_total_count")),
        "query": final_query,
        "query_seed": seed_query,
        "query_source": query_source,
        "query_variant": QUERY_VARIANT,
        "query_dspy_accepted": bool(dspy_accepted),
        "query_dspy_status": dspy_status,
        "query_fallback_used": bool(fallback_used),
        "query_new_content_tokens": pipe_join(final_audit["new_content_tokens"]),
        "query_new_content_token_count": len(final_audit["new_content_tokens"]),
        "query_evidence_source": "target_review_safe_signals_only",
        "target_metadata_fallback_used": False,
        "item_context_fallback_used": False,
        "item_metadata_evidence_used": False,
        "historical_review_evidence_used": False,
        "user_prior_evidence_used": False,
        "rating_evidence_used": False,
        "sentiment_evidence_used": False,
        "insufficient_review_evidence": False,
        "replacement_case_used": bool(row_dict["replacement_case_used"]),
        "initial_case_id": str(row_dict["initial_case_id"]),
        "replaced_case_id": str(row_dict["replaced_case_id"]),
        "replacement_source_case_id": str(row_dict["replacement_source_case_id"]),
        "query_generation_status": "generated_from_review_safe_signals",
        "query_specific_facet_family_count": int(final_audit["specific_family_count"]),
        "query_generic_anchor_terms": final_audit["generic_anchor_terms"],
        "query_generic_utility_terms": final_audit["generic_utility_terms"],
        "query_generic_anchor_rate": float(anchor_term_count / max(token_count(final_query), 1)),
        "query_specific_facet_cue_rate": float(final_audit["specific_family_count"] > 0),
        "query_clean": final_query,
        "query_clean_is_active": False,
    }
    query_rows.append(query_row)

    qc_rows.append({
        **query_row,
        "query_token_count": token_count(final_query),
        "query_seed_token_count": token_count(seed_query),
        "query_metadata_removed_terms": final_audit["removed_terms"],
        "brand_or_name_leak_flag": int(final_audit["brand_or_name_leak_flag"]),
        "identifier_like_leak_flag": int(final_audit["identifier_like_leak_flag"]),
        "asin_leak_flag": int(final_audit["asin_leak_flag"]),
        "package_cue_flag": int(final_audit["package_cue_flag"]),
        "seller_manufacturer_leak_flag": int(final_audit["seller_manufacturer_leak_flag"]),
        "exact_title_phrase_flag": int(final_audit["exact_title_phrase_flag"]),
        "query_rating_sentiment_terms": final_audit["rating_sentiment_terms"],
        "query_rating_sentiment_term_count": len(split_pipe_values(final_audit["rating_sentiment_terms"])),
        "query_missing_multiword_phrases": pipe_join(final_audit["missing_multiword_phrases"]),
        "query_missing_multiword_phrase_count": len(final_audit["missing_multiword_phrases"]),
        "title_overlap_ratio": float(final_audit["title_overlap_ratio"]),
        "title_overlap_flag": int(final_audit["title_overlap_ratio"] > TITLE_OVERLAP_WARNING_THRESHOLD),
    })

queries_df = pd.DataFrame(query_rows)
query_qc_df = pd.DataFrame(qc_rows)

print("Rows: generated", len(queries_df))
print("Validation: query generation completed")

Rows: generated 2288
Validation: query generation completed


In [11]:
# =========================================================
# Summary, Contract, and Final Validation
# =========================================================
EXPECTED_OUTPUT_ROWS = int(EXPECTED_TARGET_PER_REGIME * len(REGIME_ORDER))
expected_rows = SMOKE_TEST_N if RUN_SMOKE_TEST else EXPECTED_OUTPUT_ROWS
if len(queries_df) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows} query rows, found {len(queries_df)}.")
if queries_df["case_id"].duplicated().any():
    raise RuntimeError("case_id must be unique in the final query cache.")
if queries_df["query"].fillna("").astype(str).str.strip().eq("").any():
    raise RuntimeError("Every active query must be non-empty.")

if not RUN_SMOKE_TEST:
    final_regime_counts = (
        queries_df["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int)
    )
    if not final_regime_counts.eq(EXPECTED_TARGET_PER_REGIME).all():
        raise RuntimeError(f"Final regime quota mismatch: {final_regime_counts.to_dict()}")

required_query_audit_columns = {
    "query_evidence_source", "target_metadata_fallback_used",
    "item_context_fallback_used", "historical_review_evidence_used",
    "user_prior_evidence_used", "insufficient_review_evidence",
    "replacement_case_used", "replacement_source_case_id",
    "query_generation_status", "query_specific_facet_family_count",
    "query_generic_anchor_terms", "query_generic_utility_terms",
}
missing_query_audit_columns = sorted(required_query_audit_columns - set(queries_df.columns))
if missing_query_audit_columns:
    raise RuntimeError(f"Missing required query audit columns: {missing_query_audit_columns}")

required_false_columns = [
    "target_metadata_fallback_used", "item_context_fallback_used",
    "item_metadata_evidence_used", "historical_review_evidence_used",
    "user_prior_evidence_used", "rating_evidence_used", "sentiment_evidence_used",
    "insufficient_review_evidence", "query_clean_is_active",
]
for column in required_false_columns:
    if queries_df[column].astype(bool).any():
        raise RuntimeError(f"Final query audit field must be false: {column}")
if not queries_df["query_evidence_source"].eq("target_review_safe_signals_only").all():
    raise RuntimeError("Final query evidence source is invalid.")
if not queries_df["query_generation_status"].eq("generated_from_review_safe_signals").all():
    raise RuntimeError("Final query generation status is invalid.")
if queries_df["query_specific_facet_family_count"].lt(MIN_SPECIFIC_FAMILIES).any():
    raise RuntimeError("Final query specific-family count is below the common strict threshold.")
if queries_df["safe_signal_count"].lt(MIN_SPECIFIC_SIGNALS).any():
    raise RuntimeError("Final query source-signal count is below the common strict threshold.")
if queries_df["query_generic_anchor_terms"].map(normalize_space).ne("").any():
    raise RuntimeError("Unprotected generic category anchors remain in active queries.")
if queries_df["query_generic_utility_terms"].map(normalize_space).ne("").any():
    raise RuntimeError("Unprotected generic utility terms remain in active queries.")
if queries_df["replacement_case_used"].ne(
    queries_df["replacement_source_case_id"].map(normalize_space).ne("")
).any():
    raise RuntimeError("Replacement audit fields are inconsistent.")
replacement_rows_mask = queries_df["replacement_case_used"]
if not queries_df.loc[replacement_rows_mask, "replacement_source_case_id"].eq(
    queries_df.loc[replacement_rows_mask, "case_id"]
).all():
    raise RuntimeError("replacement_source_case_id must identify the final reserve case.")
if not queries_df.loc[replacement_rows_mask, "replaced_case_id"].eq(
    queries_df.loc[replacement_rows_mask, "initial_case_id"]
).all():
    raise RuntimeError("replaced_case_id must identify the original insufficient case.")

blocking_qc_columns = [
    "brand_or_name_leak_flag", "identifier_like_leak_flag", "asin_leak_flag",
    "package_cue_flag", "seller_manufacturer_leak_flag", "exact_title_phrase_flag",
    "query_new_content_token_count", "query_rating_sentiment_term_count",
    "query_missing_multiword_phrase_count",
]
if query_qc_df[blocking_qc_columns].fillna(0).astype(int).gt(0).any().any():
    raise RuntimeError("Final leakage, no-new-content, or multiword assertion failed.")

forbidden_output_columns = {
    "target_review_text", "heldout_review_text", "review_text", "raw_review_text",
    "query_safe_residual_text", "query_safe_facet_text", "common_functional_facet_text",
    "historical_review_reputation_text", "review_reputation_facet_text",
    "itemctx_title", "itemctx_facet_brand_text", "itemctx_identifier_diagnostic_text",
    "brand", "title", "manufacturer", "seller", "prompt", "response", "llm_response",
    "prior_review_n", "prior_history_n",
}
forbidden_output_present = sorted(forbidden_output_columns & set(queries_df.columns))
if forbidden_output_present:
    raise RuntimeError(f"Forbidden query-cache columns: {forbidden_output_present}")

comparison_rows = []
for regime in REGIME_ORDER:
    eligible_regime = pool_df[pool_df["regime"].eq(regime)]
    final_regime = queries_df[queries_df["regime"].eq(regime)]
    regime_qc = query_qc_df[query_qc_df["regime"].eq(regime)]
    comparison_rows.append({
        "category": "Facial Skincare",
        "regime": regime,
        "eligible_source_cases": int(len(eligible_regime)),
        "signal_evaluated_cases": int(eligible_regime["signal_available"].sum()),
        "unevaluated_cases": int((~eligible_regime["signal_available"]).sum()),
        "insufficient_cases": int(
            (eligible_regime["signal_available"] & ~eligible_regime["query_candidate_sufficient"]).sum()
        ),
        "initial_selected_cases": int(eligible_regime["initial_selected"].sum()),
        "initial_insufficient_cases": int(
            (eligible_regime["initial_selected"] & ~eligible_regime["query_candidate_sufficient"]).sum()
        ),
        "replacements": int(final_regime["replacement_case_used"].sum()),
        "final_cases": int(len(final_regime)),
        "generic_anchor_rate": float(
            final_regime["query_generic_anchor_terms"].map(normalize_space).ne("").mean()
        ) if len(final_regime) else 0.0,
        "specific_cue_rate": float(
            final_regime["query_specific_facet_family_count"].gt(0).mean()
        ) if len(final_regime) else 0.0,
        "leakage_assertions_passed": bool(
            regime_qc[blocking_qc_columns].fillna(0).astype(int).eq(0).all().all()
        ),
    })
comparison_df = pd.DataFrame(comparison_rows)

query_contract = {
    "category": "Facial Skincare",
    "evidence_scope": "target_review_safe_signals_only",
    "active_query_column": "query",
    "compatibility_query_aliases": {},
    "query_evidence_source": "target_review_safe_signals_only",
    "target_metadata_role": "post_generation_leakage_detection_and_scrubbing_only",
    "target_metadata_fallback_used": False,
    "item_context_fallback_used": False,
    "item_metadata_evidence_used": False,
    "historical_review_evidence_used": False,
    "user_prior_evidence_used": False,
    "rating_evidence_used": False,
    "sentiment_evidence_used": False,
    "raw_target_review_loaded": False,
    "raw_review_text_exported": False,
    "review_safe_seed_built_before_metadata_join": True,
    "user_prior_columns_loaded": [],
    "user_prior_columns_exported": [],
    "replacement_source_case_id_semantics": "final_same_regime_reserve_case_id; equals case_id on replacement rows",
    "replaced_case_id_semantics": "original_insufficient_selected_case_id",
    "replacement_policy": "same_regime_case_id_order_over_signal_ready_reserve",
    "reserve_signal_requirement": "Notebook 05 signals must exist for replacement candidates",
    "generic_anchor_rescue_enabled": False,
    "item_context_seed_enabled": False,
    "multiword_entity_preservation_required": True,
    "query_clean_column": "query_clean",
    "query_clean_is_active": False,
    "final_min_query_tokens": MIN_QUERY_TOKENS,
    "final_max_query_tokens": MAX_QUERY_TOKENS,
    "final_min_query_safe_signal_families": MIN_SPECIFIC_FAMILIES,
    "final_min_query_safe_signal_total": MIN_SPECIFIC_SIGNALS,
    "regime_order": REGIME_ORDER,
    "eligible_regime_counts": EXPECTED_ELIGIBLE_REGIME_COUNTS,
    "initial_target_per_regime": EXPECTED_INITIAL_TARGET_PER_REGIME,
    "target_per_regime": EXPECTED_TARGET_PER_REGIME,
    "query_variant": QUERY_VARIANT,
    "dspy_mode": DSPY_MODE,
    "dspy_model": DEEPSEEK_MODEL,
    "dspy_temperature": DSPY_TEMPERATURE,
    "dspy_max_tokens": DSPY_MAX_TOKENS,
    "dspy_max_retries": DSPY_MAX_RETRIES,
    "runtime_comparison": comparison_df.to_dict(orient="records"),
    "output_paths": {
        "query_cache": str(QUERY_CACHE_PATH),
        "query_qc": str(QUERY_QC_PATH),
        "contract": str(QUERY_CONTRACT_PATH),
        "insufficient_evidence_audit": str(OUTPUT_DIR / "face_query_insufficient_review_evidence.csv"),
        "replacement_mapping": str(OUTPUT_DIR / "face_query_replacement_mapping.csv"),
        "comparison": str(OUTPUT_DIR / "face_query_generation_comparison.csv"),
    },
}

print("Validation: PASS")


Validation: PASS


In [12]:
# =========================================================
# Export
# =========================================================
should_write_outputs = (not RUN_SMOKE_TEST) or WRITE_OUTPUTS_IN_SMOKE_TEST
if should_write_outputs:
    QUERY_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    queries_df.to_parquet(QUERY_CACHE_PATH, index=False)
    query_qc_df.to_parquet(QUERY_QC_PATH, index=False)
    query_qc_df.to_csv(
        OUTPUT_DIR / "face_query_generation_qc.csv",
        index=False,
        encoding="utf-8-sig",
    )
    comparison_df.to_csv(
        OUTPUT_DIR / "face_query_generation_comparison.csv",
        index=False,
        encoding="utf-8-sig",
    )
    queries_df.head(100).to_csv(
        OUTPUT_DIR / "face_queries_preview_100.csv",
        index=False,
        encoding="utf-8-sig",
    )
    queries_df[[
        "case_id", "query", "query_clean", "query_clean_is_active",
        "query_generic_anchor_terms", "query_generic_utility_terms",
    ]].to_csv(
        OUTPUT_DIR / "face_query_clean_audit.csv",
        index=False,
        encoding="utf-8-sig",
    )
    with open(QUERY_CONTRACT_PATH, "w", encoding="utf-8") as file:
        json.dump(query_contract, file, ensure_ascii=False, indent=2)

    expected_outputs = [QUERY_CACHE_PATH, QUERY_QC_PATH, QUERY_CONTRACT_PATH]
    missing_outputs = [str(path) for path in expected_outputs if not path.exists()]
    if missing_outputs:
        raise RuntimeError(f"Missing query-generation outputs: {missing_outputs}")

print("Output:", QUERY_CACHE_PATH)
print("Output:", QUERY_QC_PATH)
print("Rows:", len(queries_df))

Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_summary/face_query_generation_qc.parquet
Rows: 2288
